In [5]:
import torch
import torch.nn as nn

In [6]:
# ── The exact same 4×4 input image from our hand calculation ──────────
x = torch.tensor([[2., 1., 0., 3.],
                   [1., 0., 2., 1.],
                   [3., 2., 1., 0.],
                   [0., 1., 3., 2.]])

# PyTorch conv layers expect input shape: (batch, channels, height, width)
x = x.unsqueeze(0).unsqueeze(0)   # shape becomes (1, 1, 4, 4)



In [7]:
# ── Convolution layer: 1 input channel, 2 output channels (our 2 filters), 3×3 kernel ──
conv = nn.Conv2d(in_channels=1, out_channels=2, kernel_size=3, stride=1, padding=0, bias=False)

# Manually set the weights to match Filter A and Filter B exactly
with torch.no_grad():
    conv.weight[0, 0] = torch.tensor([[1., 0., -1.],
                                       [1., 0., -1.],
                                       [1., 0., -1.]])   # Filter A
    conv.weight[1, 0] = torch.tensor([[1.,  1.,  1.],
                                       [0.,  0.,  0.],
                                       [-1., -1., -1.]]) # Filter B

conv_out = conv(x)
print("Conv output shape:", conv_out.shape)   # expect (1, 2, 2, 2)
print("Conv output:\n", conv_out)


Conv output shape: torch.Size([1, 2, 2, 2])
Conv output:
 tensor([[[[ 3., -1.],
          [-2.,  0.]],

         [[-3.,  1.],
          [-1., -3.]]]], grad_fn=<ConvolutionBackward0>)


In [8]:
# ── ReLU ────────────────────────────────────────────────────────────
relu = nn.ReLU()
relu_out = relu(conv_out)
print("\nReLU output:\n", relu_out)

# ── Max pooling: 2×2 window, stride 2 ───────────────────────────────
pool = nn.MaxPool2d(kernel_size=2, stride=2)
pool_out = pool(relu_out)
print("\nPooled output:\n", pool_out)

# ── Flatten ──────────────────────────────────────────────────────────
flatten = nn.Flatten()
flat_out = flatten(pool_out)
print("\nFlattened vector:", flat_out)


ReLU output:
 tensor([[[[3., 0.],
          [0., 0.]],

         [[0., 1.],
          [0., 0.]]]], grad_fn=<ReluBackward0>)

Pooled output:
 tensor([[[[3.]],

         [[1.]]]], grad_fn=<MaxPool2DWithIndicesBackward0>)

Flattened vector: tensor([[3., 1.]], grad_fn=<ViewBackward0>)


In [9]:
# ── Fully-connected layer: 2 inputs → 1 output ──────────────────────
fc = nn.Linear(in_features=2, out_features=1)

with torch.no_grad():
    fc.weight[0] = torch.tensor([0.5, -1.0])
    fc.bias[0] = 0.2

z = fc(flat_out)
print("\nz (pre-activation):", z)

# ── Sigmoid ──────────────────────────────────────────────────────────
sigmoid = nn.Sigmoid()
y_hat = sigmoid(z)
print("Final prediction (ŷ):", y_hat)


z (pre-activation): tensor([[0.7000]], grad_fn=<AddmmBackward0>)
Final prediction (ŷ): tensor([[0.6682]], grad_fn=<SigmoidBackward0>)
